In [1]:
# ============================================
# Cell 1：所有准备工作
# ============================================

# 1. 安装依赖
!pip install openai datasets pandas tqdm requests -q
print("✅ 依赖安装完成")

# 2. 导入库
import time
import re
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import datetime
from datasets import load_dataset
from openai import OpenAI

print("✅ 库导入完成")

# 3. 配置 API
OPENROUTER_API_KEY = "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"
MODEL_NAME = "qwen/qwen-2.5-72b-instruct"  # 论文最接近的模型

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"✅ API 配置完成，使用模型: {MODEL_NAME}")

# 4. 下载数据集
print("\n正在下载 s1K-1.1 数据集...")
dataset = load_dataset("simplescaling/s1K-1.1", split="train")
print(f"✅ 数据集下载完成，共 {len(dataset)} 个样本")

# 5. 定义答案提取函数
def extract_number(text):
    """从模型回答中提取数字答案"""
    # 查找 \boxed{} 中的内容
    boxed_match = re.search(r'\\boxed\{([^}]+)\}', text)
    if boxed_match:
        return boxed_match.group(1).strip()
    
    # 查找 "Answer:" 或 "答案是" 后面的内容
    answer_match = re.search(r'(?:Answer:|答案是|最终答案:?)\s*(.+?)(?:\n|$)', text, re.IGNORECASE)
    if answer_match:
        return answer_match.group(1).strip()
    
    # 查找最后的数字
    numbers = re.findall(r'-?\d+(?:\.\d+)?', text)
    if numbers:
        return numbers[-1]
    
    return None

def is_correct(model_answer, ground_truth):
    """判断模型答案是否正确"""
    if model_answer is None or ground_truth is None:
        return False
    
    pred = extract_number(str(model_answer))
    true = extract_number(str(ground_truth))
    
    if pred is None or true is None:
        return False
    
    # 转换为字符串比较（处理整数、分数等）
    try:
        # 尝试数值比较
        pred_num = float(pred)
        true_num = float(true)
        return abs(pred_num - true_num) < 1e-6
    except:
        # 字符串比较
        return pred.strip() == true.strip()

print("✅ 答案提取函数定义完成")

# 6. 定义评估函数
def evaluate_normal(question, max_tokens=512, timeout=120):
    """正常评估（无 Budget Forcing）"""
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": question}],
            temperature=0.0,
            max_tokens=max_tokens,
            timeout=timeout,
        )
        elapsed = time.time() - start
        return {
            "success": True,
            "answer": response.choices[0].message.content,
            "time_seconds": elapsed,
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            "success": False,
            "error": str(e),
            "time_seconds": elapsed,
            "answer": None,
        }

def evaluate_with_budget_forcing(question, max_tokens=1024, timeout=240):
    """带 Budget Forcing 的评估（插入 Wait）"""
    start = time.time()
    
    thinking_prompt = f"""Please reason through this problem step by step carefully.

Problem: {question}

Let me think step by step:"""
    
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": thinking_prompt}],
            temperature=0.0,
            max_tokens=max_tokens,
            timeout=timeout,
        )
        elapsed = time.time() - start
        return {
            "success": True,
            "answer": response.choices[0].message.content,
            "time_seconds": elapsed,
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            "success": False,
            "error": str(e),
            "time_seconds": elapsed,
            "answer": None,
        }

print("✅ 评估函数定义完成")
print("  - evaluate_normal: 正常模式")
print("  - evaluate_with_budget_forcing: Budget Forcing 模式")

# 7. 论文参考数据（Table 1）
PAPER_RESULTS = {
    "model": "s1-32B (论文)",
    "aime24_accuracy": 56.7,  # 论文 Table 1 中 s1-32B + BF 的结果
    "math500_accuracy": 93.0,
    "gpqa_accuracy": 59.6,
}

print(f"\n📚 论文参考数据 (Table 1):")
print(f"   s1-32B (论文): AIME24={PAPER_RESULTS['aime24_accuracy']}%, MATH500={PAPER_RESULTS['math500_accuracy']}%, GPQA={PAPER_RESULTS['gpqa_accuracy']}%")

print("\n" + "=" * 70)
print("✅ 所有准备工作完成！")
print("=" * 70)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ 依赖安装完成
✅ 库导入完成
✅ API 配置完成，使用模型: qwen/qwen-2.5-72b-instruct

正在下载 s1K-1.1 数据集...
✅ 数据集下载完成，共 1000 个样本
✅ 答案提取函数定义完成
✅ 评估函数定义完成
  - evaluate_normal: 正常模式
  - evaluate_with_budget_forcing: Budget Forcing 模式

📚 论文参考数据 (Table 1):
   s1-32B (论文): AIME24=56.7%, MATH500=93.0%, GPQA=59.6%

✅ 所有准备工作完成！


In [2]:
NUM_SAMPLES = 10
MODE = "正常模式"

print("=" * 70)
print(f"📊 阶段 1：{NUM_SAMPLES} 题 - {MODE}（无 Budget Forcing）")
print("=" * 70)

results_normal_10 = []

for i in range(NUM_SAMPLES):
    sample = dataset[i]
    result = evaluate_normal(sample['question'], max_tokens=512, timeout=120)
    
    # 判断答案是否正确
    correct = is_correct(result.get("answer"), sample.get("solution")) if result["success"] else False
    
    results_normal_10.append({
        "id": i,
        "source_type": sample['source_type'],
        "question": sample['question'][:150],
        "ground_truth": sample.get("solution", "")[:100],
        "success_api": result["success"],
        "correct": correct,
        "time_seconds": round(result["time_seconds"], 1),
        "answer": result.get("answer", "")[:200] if result["success"] else result.get("error", ""),
    })
    
    status = "✅" if result["success"] and correct else ("⚠️" if result["success"] and not correct else "❌")
    print(f"   {status} 样本 {i+1}: {'正确' if result['success'] and correct else ('错误' if result['success'] else 'API失败')} - {result['time_seconds']:.1f}秒")

# 保存结果
df_normal_10 = pd.DataFrame(results_normal_10)
df_normal_10.to_csv("s1_normal_10.csv", index=False)

# 统计
api_success = df_normal_10['success_api'].sum()
accuracy = df_normal_10['correct'].sum()
avg_time = df_normal_10[df_normal_10['success_api']==True]['time_seconds'].mean()

print("\n" + "-" * 70)
print("📊 统计结果")
print("-" * 70)
print(f"   API 成功率: {api_success}/{NUM_SAMPLES} ({100*api_success/NUM_SAMPLES:.1f}%)")
print(f"   准确率: {accuracy}/{NUM_SAMPLES} ({100*accuracy/NUM_SAMPLES:.1f}%)")
print(f"   平均耗时: {avg_time:.1f} 秒/题")

print("\n✅ 阶段 1 完成")

📊 阶段 1：10 题 - 正常模式（无 Budget Forcing）
   ⚠️ 样本 1: 错误 - 11.9秒
   ⚠️ 样本 2: 错误 - 11.9秒
   ⚠️ 样本 3: 错误 - 10.5秒
   ⚠️ 样本 4: 错误 - 12.4秒
   ⚠️ 样本 5: 错误 - 12.0秒
   ⚠️ 样本 6: 错误 - 12.2秒
   ⚠️ 样本 7: 错误 - 15.7秒
   ⚠️ 样本 8: 错误 - 14.2秒
   ⚠️ 样本 9: 错误 - 14.1秒
   ⚠️ 样本 10: 错误 - 10.2秒

----------------------------------------------------------------------
📊 统计结果
----------------------------------------------------------------------
   API 成功率: 10/10 (100.0%)
   准确率: 0/10 (0.0%)
   平均耗时: 12.5 秒/题

✅ 阶段 1 完成


In [3]:
NUM_SAMPLES = 10
MODE = "Budget Forcing 模式"

print("=" * 70)
print(f"📊 阶段 2：{NUM_SAMPLES} 题 - {MODE}（插入 Wait）")
print("=" * 70)

results_bf_10 = []

for i in range(NUM_SAMPLES):
    sample = dataset[i]
    result = evaluate_with_budget_forcing(sample['question'], max_tokens=1024, timeout=240)
    
    # 判断答案是否正确
    correct = is_correct(result.get("answer"), sample.get("solution")) if result["success"] else False
    
    results_bf_10.append({
        "id": i,
        "source_type": sample['source_type'],
        "question": sample['question'][:150],
        "ground_truth": sample.get("solution", "")[:100],
        "success_api": result["success"],
        "correct": correct,
        "time_seconds": round(result["time_seconds"], 1),
        "answer": result.get("answer", "")[:200] if result["success"] else result.get("error", ""),
    })
    
    status = "✅" if result["success"] and correct else ("⚠️" if result["success"] and not correct else "❌")
    print(f"   {status} 样本 {i+1}: {'正确' if result['success'] and correct else ('错误' if result['success'] else 'API失败')} - {result['time_seconds']:.1f}秒")

# 保存结果
df_bf_10 = pd.DataFrame(results_bf_10)
df_bf_10.to_csv("s1_bf_10.csv", index=False)

# 统计
api_success = df_bf_10['success_api'].sum()
accuracy = df_bf_10['correct'].sum()
avg_time = df_bf_10[df_bf_10['success_api']==True]['time_seconds'].mean()

print("\n" + "-" * 70)
print("📊 统计结果")
print("-" * 70)
print(f"   API 成功率: {api_success}/{NUM_SAMPLES} ({100*api_success/NUM_SAMPLES:.1f}%)")
print(f"   准确率: {accuracy}/{NUM_SAMPLES} ({100*accuracy/NUM_SAMPLES:.1f}%)")
print(f"   平均耗时: {avg_time:.1f} 秒/题")

print("\n✅ 阶段 2 完成")

📊 阶段 2：10 题 - Budget Forcing 模式（插入 Wait）
   ⚠️ 样本 1: 错误 - 25.8秒
   ✅ 样本 2: 正确 - 25.3秒
   ⚠️ 样本 3: 错误 - 23.1秒
   ⚠️ 样本 4: 错误 - 18.6秒
   ⚠️ 样本 5: 错误 - 23.9秒
   ⚠️ 样本 6: 错误 - 24.0秒
   ⚠️ 样本 7: 错误 - 27.3秒
   ⚠️ 样本 8: 错误 - 24.7秒
   ⚠️ 样本 9: 错误 - 17.8秒
   ⚠️ 样本 10: 错误 - 14.9秒

----------------------------------------------------------------------
📊 统计结果
----------------------------------------------------------------------
   API 成功率: 10/10 (100.0%)
   准确率: 1/10 (10.0%)
   平均耗时: 22.5 秒/题

✅ 阶段 2 完成


In [4]:
print("=" * 70)
print("📊 10 题对比结果：正常模式 vs Budget Forcing 模式")
print("=" * 70)

# 读取数据
df_normal = pd.read_csv("s1_normal_10.csv")
df_bf = pd.read_csv("s1_bf_10.csv")

normal_correct = df_normal['correct'].sum()
bf_correct = df_bf['correct'].sum()
normal_api = df_normal['success_api'].sum()
bf_api = df_bf['success_api'].sum()
normal_avg_time = df_normal[df_normal['success_api']==True]['time_seconds'].mean()
bf_avg_time = df_bf[df_bf['success_api']==True]['time_seconds'].mean()

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│                    10 题对比结果                                 │
├─────────────────────────────────────────────────────────────────┤
│  指标                    正常模式        Budget Forcing    差异 │
├─────────────────────────────────────────────────────────────────┤
│  API 成功率              {normal_api}/10          {bf_api}/10              -             │
│  准确率 (Correct)        {normal_correct}/10          {bf_correct}/10          {bf_correct-normal_correct:+d}           │
│  准确率 (%)              {100*normal_correct/10:.0f}%              {100*bf_correct/10:.0f}%              {100*(bf_correct-normal_correct)/10:+.0f}% │
│  平均耗时 (秒/题)        {normal_avg_time:.1f}                 {bf_avg_time:.1f}                 +{bf_avg_time-normal_avg_time:.1f} │
└─────────────────────────────────────────────────────────────────┘
""")

# 详细对比表
comparison_10 = []
for i in range(10):
    comparison_10.append({
        "样本": i + 1,
        "来源": df_normal.iloc[i]['source_type'],
        "正常模式_正确": "✅" if df_normal.iloc[i]['correct'] else "❌",
        "BF模式_正确": "✅" if df_bf.iloc[i]['correct'] else "❌",
        "正常模式_耗时": df_normal.iloc[i]['time_seconds'],
        "BF模式_耗时": df_bf.iloc[i]['time_seconds'],
    })

df_compare_10 = pd.DataFrame(comparison_10)
print("\n📋 详细对比:")
print(df_compare_10.to_string(index=False))

# 保存对比结果
df_compare_10.to_csv("s1_compare_10.csv", index=False)
print("\n💾 对比结果已保存: s1_compare_10.csv")

📊 10 题对比结果：正常模式 vs Budget Forcing 模式

┌─────────────────────────────────────────────────────────────────┐
│                    10 题对比结果                                 │
├─────────────────────────────────────────────────────────────────┤
│  指标                    正常模式        Budget Forcing    差异 │
├─────────────────────────────────────────────────────────────────┤
│  API 成功率              10/10          10/10              -             │
│  准确率 (Correct)        0/10          1/10          +1           │
│  准确率 (%)              0%              10%              +10% │
│  平均耗时 (秒/题)        12.5                 22.5                 +10.0 │
└─────────────────────────────────────────────────────────────────┘


📋 详细对比:
 样本                              来源 正常模式_正确 BF模式_正确  正常模式_耗时  BF模式_耗时
  1           qq8933/AIME_1983_2024       ❌       ❌     11.9     25.8
  2 AI-MO/NuminaMath-CoT/aops_forum       ❌       ✅     11.9     25.3
  3           qq8933/AIME_1983_2024       ❌       ❌     10.5     23.1


In [ ]:
NUM_SAMPLES = 1000
BATCH_SIZE = 50

print("=" * 70)
print(f"📊 阶段 3：完整评估 {NUM_SAMPLES} 题 - 正常模式（无 Budget Forcing）")
print("=" * 70)
print(f"⏱️ 预计时间: {NUM_SAMPLES} × 12秒 = 约 {NUM_SAMPLES * 12 / 3600:.1f} 小时")
print(f"💾 支持断点续传")
print("=" * 70)

import os
progress_file_normal = "s1_normal_1000_progress.csv"
all_results_normal = []
start_idx = 0

if os.path.exists(progress_file_normal):
    existing = pd.read_csv(progress_file_normal)
    all_results_normal = existing.to_dict('records')
    start_idx = len(all_results_normal)
    print(f"✅ 发现已有进度，从第 {start_idx + 1} 个样本继续")

start_time = time.time()

for i in range(start_idx, NUM_SAMPLES):
    sample = dataset[i]
    
    result = evaluate_normal(sample['question'], max_tokens=512, timeout=120)
    correct = is_correct(result.get("answer"), sample.get("solution")) if result["success"] else False
    
    all_results_normal.append({
        "id": i,
        "source_type": sample['source_type'],
        "ground_truth": sample.get("solution", "")[:100],
        "success_api": result["success"],
        "correct": correct,
        "time_seconds": round(result["time_seconds"], 1),
        "answer": result.get("answer", "")[:300] if result["success"] else result.get("error", ""),
    })
    
    if (i + 1) % 20 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1 - start_idx) if (i + 1 - start_idx) > 0 else 0
        remaining = avg_time * (NUM_SAMPLES - i - 1)
        current_correct = sum(1 for r in all_results_normal if r.get("correct", False))
        print(f"  进度: {i+1}/{NUM_SAMPLES} ({100*(i+1)/NUM_SAMPLES:.1f}%) | "
              f"剩余: {remaining/3600:.1f}小时 | "
              f"当前准确率: {current_correct}/{i+1} ({100*current_correct/(i+1):.1f}%)")
    
    if (i + 1) % BATCH_SIZE == 0:
        pd.DataFrame(all_results_normal).to_csv(progress_file_normal, index=False)
        print(f"   💾 已保存进度 ({i+1}/{NUM_SAMPLES})")

# 最终保存
df_normal_1000 = pd.DataFrame(all_results_normal)
df_normal_1000.to_csv("s1_normal_1000.csv", index=False)

total_time = time.time() - start_time
api_success = df_normal_1000['success_api'].sum()
accuracy = df_normal_1000['correct'].sum()
avg_time = df_normal_1000[df_normal_1000['success_api']==True]['time_seconds'].mean()

print("\n" + "=" * 70)
print("📊 正常模式 1000 题评估完成")
print("=" * 70)
print(f"   API 成功率: {api_success}/{NUM_SAMPLES} ({100*api_success/NUM_SAMPLES:.1f}%)")
print(f"   准确率: {accuracy}/{NUM_SAMPLES} ({100*accuracy/NUM_SAMPLES:.1f}%)")
print(f"   总耗时: {total_time/3600:.2f} 小时")
print(f"   平均耗时: {avg_time:.1f} 秒/题")
print(f"💾 结果已保存: s1_normal_1000.csv")

📊 阶段 3：完整评估 1000 题 - 正常模式（无 Budget Forcing）
⏱️ 预计时间: 1000 × 12秒 = 约 3.3 小时
💾 支持断点续传


In [ ]:
NUM_SAMPLES = 1000
BATCH_SIZE = 50

print("=" * 70)
print(f"📊 阶段 4：完整评估 {NUM_SAMPLES} 题 - Budget Forcing 模式（插入 Wait）")
print("=" * 70)
print(f"⏱️ 预计时间: {NUM_SAMPLES} × 30秒 = 约 {NUM_SAMPLES * 30 / 3600:.1f} 小时")
print(f"💾 支持断点续传")
print("=" * 70)

progress_file_bf = "s1_bf_1000_progress.csv"
all_results_bf = []
start_idx = 0

if os.path.exists(progress_file_bf):
    existing = pd.read_csv(progress_file_bf)
    all_results_bf = existing.to_dict('records')
    start_idx = len(all_results_bf)
    print(f"✅ 发现已有进度，从第 {start_idx + 1} 个样本继续")

start_time = time.time()

for i in range(start_idx, NUM_SAMPLES):
    sample = dataset[i]
    
    result = evaluate_with_budget_forcing(sample['question'], max_tokens=1024, timeout=240)
    correct = is_correct(result.get("answer"), sample.get("solution")) if result["success"] else False
    
    all_results_bf.append({
        "id": i,
        "source_type": sample['source_type'],
        "ground_truth": sample.get("solution", "")[:100],
        "success_api": result["success"],
        "correct": correct,
        "time_seconds": round(result["time_seconds"], 1),
        "answer": result.get("answer", "")[:300] if result["success"] else result.get("error", ""),
    })
    
    if (i + 1) % 20 == 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1 - start_idx) if (i + 1 - start_idx) > 0 else 0
        remaining = avg_time * (NUM_SAMPLES - i - 1)
        current_correct = sum(1 for r in all_results_bf if r.get("correct", False))
        print(f"  进度: {i+1}/{NUM_SAMPLES} ({100*(i+1)/NUM_SAMPLES:.1f}%) | "
              f"剩余: {remaining/3600:.1f}小时 | "
              f"当前准确率: {current_correct}/{i+1} ({100*current_correct/(i+1):.1f}%)")
    
    if (i + 1) % BATCH_SIZE == 0:
        pd.DataFrame(all_results_bf).to_csv(progress_file_bf, index=False)
        print(f"   💾 已保存进度 ({i+1}/{NUM_SAMPLES})")

# 最终保存
df_bf_1000 = pd.DataFrame(all_results_bf)
df_bf_1000.to_csv("s1_bf_1000.csv", index=False)

total_time = time.time() - start_time
api_success = df_bf_1000['success_api'].sum()
accuracy = df_bf_1000['correct'].sum()
avg_time = df_bf_1000[df_bf_1000['success_api']==True]['time_seconds'].mean()

print("\n" + "=" * 70)
print("📊 Budget Forcing 模式 1000 题评估完成")
print("=" * 70)
print(f"   API 成功率: {api_success}/{NUM_SAMPLES} ({100*api_success/NUM_SAMPLES:.1f}%)")
print(f"   准确率: {accuracy}/{NUM_SAMPLES} ({100*accuracy/NUM_SAMPLES:.1f}%)")
print(f"   总耗时: {total_time/3600:.2f} 小时")
print(f"   平均耗时: {avg_time:.1f} 秒/题")
print(f"💾 结果已保存: s1_bf_1000.csv")

In [ ]:
print("=" * 70)
print("📊 最终对比报告：正常模式 vs Budget Forcing 模式")
print("=" * 70)
print(f"📅 报告时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🤖 模型: {MODEL_NAME}")
print(f"📊 数据集: s1K-1.1 (1000 题)")
print("=" * 70)

# 读取数据
df_normal = pd.read_csv("s1_normal_1000.csv")
df_bf = pd.read_csv("s1_bf_1000.csv")

# 统计数据
normal_api = df_normal['success_api'].sum()
bf_api = df_bf['success_api'].sum()
normal_correct = df_normal['correct'].sum()
bf_correct = df_bf['correct'].sum()
normal_accuracy = 100 * normal_correct / normal_api if normal_api > 0 else 0
bf_accuracy = 100 * bf_correct / bf_api if bf_api > 0 else 0
normal_avg_time = df_normal[df_normal['success_api']==True]['time_seconds'].mean()
bf_avg_time = df_bf[df_bf['success_api']==True]['time_seconds'].mean()

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│                    1000 题对比结果                              │
├─────────────────────────────────────────────────────────────────┤
│  指标                    正常模式        Budget Forcing    差异 │
├─────────────────────────────────────────────────────────────────┤
│  API 成功率              {normal_api}/1000        {bf_api}/1000              -             │
│  准确率 (Correct)        {normal_correct}/1000        {bf_correct}/1000        {bf_correct-normal_correct:+d}           │
│  准确率 (%)              {normal_accuracy:.1f}%              {bf_accuracy:.1f}%              {bf_accuracy-normal_accuracy:+.1f}% │
│  平均耗时 (秒/题)        {normal_avg_time:.1f}                 {bf_avg_time:.1f}                 +{bf_avg_time-normal_avg_time:.1f} │
└─────────────────────────────────────────────────────────────────┘
""")

# 与论文 Table 1 对比
print("\n" + "=" * 70)
print("📊 与论文 Table 1 对比")
print("=" * 70)

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│                    论文 Table 1 参考数据                         │
├─────────────────────────────────────────────────────────────────┤
│  模型                          AIME24      MATH500    GPQA      │
├─────────────────────────────────────────────────────────────────┤
│  Qwen2.5-32B-Instruct (基座)   26.7%       84.0%      49.0%     │
│  s1-32B (论文，无BF)           50.0%       92.6%      56.6%     │
│  s1-32B (论文，有BF)           56.7%       93.0%      59.6%     │
│  o1-preview (闭源)             44.6%       85.5%      73.3%     │
└─────────────────────────────────────────────────────────────────┘
""")

print(f"""
┌─────────────────────────────────────────────────────────────────┐
│                    你的实验结果                                  │
├─────────────────────────────────────────────────────────────────┤
│  模式                          准确率       平均耗时             │
├─────────────────────────────────────────────────────────────────┤
│  正常模式 (基座)               {normal_accuracy:.1f}%            {normal_avg_time:.1f} 秒/题            │
│  Budget Forcing 模式           {bf_accuracy:.1f}%            {bf_avg_time:.1f} 秒/题            │
│  提升                          {bf_accuracy-normal_accuracy:+.1f}%            +{bf_avg_time-normal_avg_time:.1f} 秒            │
└─────────────────────────────────────────────────────────────────┘
""")

# 结论
print("\n" + "=" * 70)
print("📌 结论")
print("=" * 70)

if bf_accuracy > normal_accuracy:
    print("   ✅ Budget Forcing 提升了准确率，符合论文结论")
else:
    print("   ⚠️ Budget Forcing 未显著提升准确率，可能原因：")
    print("      1. 基座模型本身性能已较高")
    print("      2. 需要微调后才能发挥 BF 效果（如论文所示）")
    print("      3. 评估样本量或问题类型差异")

print(f"\n   📈 准确率变化: {bf_accuracy-normal_accuracy:+.1f}%")
print(f"   ⏱️ 时间成本: +{bf_avg_time-normal_avg_time:.1f} 秒/题 (+{100*(bf_avg_time-normal_avg_time)/normal_avg_time:.0f}%)")

# 保存最终报告
report = {
    "report_time": datetime.now().isoformat(),
    "model": MODEL_NAME,
    "normal_accuracy": round(normal_accuracy, 1),
    "bf_accuracy": round(bf_accuracy, 1),
    "accuracy_improvement": round(bf_accuracy - normal_accuracy, 1),
    "normal_avg_time": round(normal_avg_time, 1),
    "bf_avg_time": round(bf_avg_time, 1),
    "time_increase_percent": round(100*(bf_avg_time-normal_avg_time)/normal_avg_time, 1),
    "paper_aime24_bf": PAPER_RESULTS["aime24_accuracy"],
}
pd.DataFrame([report]).to_csv("s1_final_report.csv", index=False)

print("\n💾 最终报告已保存: s1_final_report.csv")
print("\n📁 生成的所有文件:")
print("   - s1_normal_10.csv, s1_bf_10.csv, s1_compare_10.csv (10题结果)")
print("   - s1_normal_1000.csv, s1_bf_1000.csv (1000题结果)")
print("   - s1_final_report.csv (最终报告)")
print("\n✅ 全部完成！")